# Pretrained lens summary with generated-answer correctness

**Question:** Does J-Lens retrieve intermediates better than the logit lens, including on
items where the model completes the answer correctly across multiple tokens?
**Answer/status:** The offline scoring examples below are executed. Gemma results are
**not measured here yet**; run the GPU cells and save their outputs. No model results from
another notebook are carried over. This notebook loads a pretrained lens and never fits.

Default: **google/gemma-4-31B**, bf16, explicit BOS, and its matching Neuronpedia lens.
Use a large-memory GPU (the user's run used an A100 80 GB). Change model and lens together;
for Qwen base, set `BOS_POLICY="none"`. Lens evaluation is batched; answer generation uses
length-sorted, left-padded greedy batches with KV caching. Both use the same prepared prompts.

We keep whole-token lens ranks separate from completed-answer correctness: `14` and
`fourteen` can both be correct even when neither digit string is a single vocabulary token.
Generation and lens tables have separate persistent caches. Rescoring requires no inference.


## Setup and one configuration

Run setup first, including on fresh Colab. It finds a local checkout or clones this fork,
then installs its dependencies on Colab; locally use `INSTALL_DEPENDENCIES=True` if needed.
Use a checkout containing the batched evaluators and summary helpers. If upgrading packages
already imported into a kernel, restart it before proceeding. Installation does not load weights.
Caches are persistent files under `CACHE_DIR` (Colab's local disk does **not** survive runtime
replacement; point it to an already-mounted persistent location if desired).
After changing model/revision/lens/dtype/device/BOS, rerun **load → metadata/cache → evaluation
→ explorer**. Reload invalidates provenance immediately, closes old explorer callbacks, and
releases notebook-owned model references before allocation; configuration and disk caches
are preserved. Manually retained model aliases or debugger/output references must also be
released (restart the kernel if necessary). Cache access rejects missing or changed live state.
For this notebook the checkout must also contain `jlens/generated_answers.py`. Lens caches
and generation caches are independent. After changing generation settings rerun generation,
scoring and analysis; after changing aliases rerun scoring and analysis only.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/ai360-project-school-j-lens/jacobian-lens-gpt2.git"
INSTALL_DEPENDENCIES = "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules
REPO_DIR = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "jlens" / "evaluation.py").is_file()), None,
)
if REPO_DIR is None:
    REPO_DIR = Path.cwd() / "jacobian-lens-gpt2"
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
REPO_DIR = REPO_DIR.resolve()
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR))
if not (REPO_DIR / "jlens" / "generated_answers.py").is_file():
    raise RuntimeError("Update this checkout to the revision containing the generated-answer notebook.")


In [ ]:
import gc
import gzip
import hashlib
import inspect
import json
import pickle
import tempfile
import urllib.parse
import urllib.request
import weakref

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import transformers
from IPython.display import clear_output, display
from transformers import AutoModelForCausalLM, AutoTokenizer

import jlens
from jlens.batched_evaluation import (
    evaluate_distributions_batched,
    evaluate_paired_batched,
)
from jlens.evaluation import DATASETS, load_eval
from jlens.generated_answers import (
    SCORING_VERSION,
    generate_answers_batched,
    generated_answer_counts,
    greedy_generation_config,
    score_answer,
    score_generations,
)
from jlens.reference_plots import (
    intermediate_rank_sweep,
    plot_distribution_summary,
    plot_intermediate_sweep,
    plot_lens_comparison,
)
from jlens.strict_scoring import DecodedSpellings, ExplicitPromptModel
from jlens.summaries import (
    answer_counts,
    plot_retrieval_summary,
    retrieval_summary,
    target_final_counts,
)
from jlens.vis import build_page, compute_slice, notebook_iframe


In [ ]:
MODEL_ID = "google/gemma-4-31B"
MODEL_REVISION = "main"  # Prefer an immutable model/tokenizer commit.
LENS_URL = "https://huggingface.co/neuronpedia/jacobian-lens/resolve/main/gemma-4-31b/jlens/Salesforce-wikitext/gemma-4-31B_jacobian_lens.pt"
LOCAL_LENS_PATH = None  # Existing trusted .pt file; overrides the URL.
CACHE_DIR = Path.home() / ".cache" / "jlens-pretrained-summary"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16  # Gemma 31B weights alone need roughly 62 GB.
BOS_POLICY = "prepend"  # Gemma. Use "none" for Qwen base.
K = 10
BATCH_OPTIONS = dict(
    batch_size=8, max_batch_tokens=2048, max_seq_len=512,
    position_chunk_size=8, rank_chunk_size=8, batching="auto",
)
# Reduce batch/token budgets if activations OOM; reduce chunks for vocab workspace.
# auto uses mask-safe right padding, or exact-length batches without padding.
USE_CACHES = True  # Read only your own trusted pickle caches.
REFRESH_LENS = False  # Re-download a mutable URL; local lens files are never deleted.
REFRESH_TASKS = False
REFRESH_HELD_OUT = False
REFRESH_GEOMETRY = False
REFRESH_SLICES = False

# Independently supplied passages only; never substitute the task prompts.
HELD_OUT_TEXTS = None  # e.g. [Path("/path/to/independent.txt").read_text()]
HELD_OUT_SOURCE = ""  # Corpus/version, split, selection, and independence rationale.
HELD_OUT_OPTIONS = dict(
    batch_size=8, max_batch_tokens=1024, max_seq_len=128,
    position_chunk_size=4, batching="auto",
)
GEOMETRY_VOCAB_CHUNK_SIZE = 1024
# No silent truncation: shorten/split overlength texts explicitly and record how.
ENABLE_QWEN_GLOSS = False  # Optional unverified display-only mapping; never scoring.
# Same explicit preprocessing for lens and generation. Newlines remain intact.
TASK_PROMPT_POLICY = "trim_completion_spaces"  # Or "preserve".
GENERATION_OPTIONS = dict(
    batch_size=8, max_batch_tokens=2048, max_seq_len=512, max_new_tokens=32,
)
REFRESH_GENERATIONS = False
# Explicit complete-answer equivalents; empty means no additional aliases.
# Example: {"multihop": {"Atlantic": ["Atlantic Ocean"]}}
ANSWER_ALIASES = {}


### Offline scoring examples (no weights or GPU)

These saved examples test the extraction contract. They do not claim model performance.
The final row shows that hitting the token cap without finishing an answer is not a match.


In [1]:
examples = [
    ("14.", "14", "order-ops", False),
    ("fourteen\n", "14", "order-ops", False),
    ("14", "fourteen", "order-ops", False),
    ("1.", "14", "order-ops", False),
    ("140.", "14", "order-ops", False),
    ("15. Actually, fourteen.", "14", "order-ops", False),
    ('pequeño"', "pequeño", "multilingual", False),
    ("New York.", "New York", "multihop", False),
    ("Tirana.", "Iran", "multihop", False),
    ("14", "14", "order-ops", True),
]
example_results = pd.DataFrame([
    {"continuation": repr(text), "target": target, **score_answer(
        text, target, dataset=dataset, truncated=truncated,
    )}
    for text, target, dataset, truncated in examples
])
assert example_results.answer_correct.tolist() == [True, True, True, False, False, False, True, True, False, False]
display(example_results)


continuation,target,generated_answer,normalized_answer,answer_correct,answer_status
'14.',14,14,14,True,match
'fourteen\n',14,fourteen,fourteen,True,match
'14',fourteen,14,14,True,match
'1.',14,1,1,False,wrong_answer
'140.',14,140,140,False,wrong_answer
"'15. Actually, fourteen.'",14,15,15,False,wrong_answer
"'pequeño""'",pequeño,pequeño,pequeño,True,match
'New York.',New York,New York,new york,True,match
'Tirana.',Iran,Tirana,tirana,False,wrong_answer
'14',14,14,14,False,truncated_answer


### Protocol and provenance

* For multihop, multilingual and arithmetic, `trim_completion_spaces` removes only trailing
  ASCII spaces/tabs before either evaluation. `preserve` disables this. Newlines and other
  datasets are unchanged. Original/effective prompts are displayed for audit. BOS is explicit.
* **Lexical readouts:** full-vocabulary, complete decoded token spellings, with case/space
  variants and arithmetic synonyms. Unsupported words have null ranks. Keep `model_correct`
  and its clearer alias `next_token_target_match` as the original one-token measurement.
* **Generated answers:** one greedy continuation per annotated item, including targets with
  no whole-token spelling. No chat template, target hint, answer prefill, or teacher forcing.
  The generated answer is the first phrase before punctuation, a closing quote, or a newline.
  Leading whitespace/quotes/markdown and optional `Answer:` / `The answer is` are removed.
  A phrase without a delimiter is complete only when generation reached EOS. A capped
  unfinished phrase is a miss, never accepted as a potentially truncated prefix.
* Numeric targets and arithmetic accept equivalent digits/decimals/fractions and English
  integers 0–99 (including negative forms). Text answers use Unicode NFC, case folding and
  collapsed whitespace, with explicit aliases only. No fuzzy/substring search, later-answer
  search, or semantic judge. Explanations before the answer and unlisted synonyms may fail;
  inspect raw completions and status counts before interpreting the score as model ability.
* Generated accuracy is correct / **all annotated items**, independent of vocabulary support.
  Empty, malformed and truncated answers count as misses; datasets without targets are N/A.
  Correct-only retrieval selects `answer_correct == True`, then applies the same supported
  intermediate coverage as the all-items panel. Few selected items mean weak evidence.
* `model_correct`, final token ranks and lens retrieval remain separate from answer scoring.
  Both lenses share final logits and generated answers. A lens cannot change model accuracy.
* Cache keys include prompts, model/tokenizer/BOS/precision, implementation hashes and explicit
  decoding settings. Aliases/scoring do not require regeneration. External fitting provenance
  remains unverified; pin revisions and save the executed notebook when reporting results.


In [ ]:
# Invalidate FIRST, including when this load subsequently fails.
run_metadata = _metadata_binding = _loaded_binding = None


def close_slice_explorer():
    button = globals().get("slice_run_button")
    callback = globals().get("on_slice_click")
    if button is not None:
        button.disabled = True
        if callback is not None:
            button.on_click(callback, remove=True)
    for name in ("slice_ui", "slice_output", "slice_prompt_editor", "slice_run_button"):
        widget = globals().pop(name, None)
        if widget is not None:
            widget.close()
    globals().pop("on_slice_click", None)
    globals().pop("run_slice", None)
    globals().pop("_explorer_binding", None)


close_slice_explorer()
# Drop every notebook-owned model/root alias, partial load and dependent result.
# Do not clear user configuration or delete any disk caches.
for name in (
    "model", "adapter", "hf_model", "tokenizer", "fitted_lens", "spellings",
    "candidate", "matrix", "lens_path", "_loaded_lens_sha256", "tokenizer_signature",
    "all_words", "items", "evals", "task_settings", "correct_counts",
    "token_items", "generations", "answers", "generated_counts", "generation_settings",
    "_task_binding", "_answer_binding", "prompt_audit",
    "intermediate", "answer_targets", "final_targets", "shared_final", "sweep",
    "held_out_metrics", "held_out_geometry", "held_out_run", "candidate_metrics",
    "candidate_geometry", "unavailable_geometry", "task_prompts",
    "fig", "axes", "ax", "fig52", "axes52", "fig55", "axes55", "fig56", "axes56",
):
    globals().pop(name, None)
# Exception tracebacks can retain models from failed calls in IPython/Python.
for name in ("last_traceback", "last_value", "last_exc"):
    if hasattr(sys, name):
        setattr(sys, name, None)
plt.close("all")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


def requested_load_config():
    return dict(model_id=MODEL_ID, revision=MODEL_REVISION, dtype=str(DTYPE),
                device=str(DEVICE), bos=BOS_POLICY, lens_url=LENS_URL,
                local_lens=None if LOCAL_LENS_PATH is None else str(LOCAL_LENS_PATH))


def live_signature():
    # Scalar signatures only: provenance must never keep GPU objects alive.
    def tensor_signature(tensor):
        return (id(tensor), str(tensor.dtype), str(tensor.device),
                tuple(tensor.shape), tensor._version)

    if (model.model is not adapter or adapter._hf_model is not hf_model
            or adapter.tokenizer is not tokenizer):
        raise RuntimeError("Loaded model/adapter/tokenizer links changed.")
    return {
        "requested": requested_load_config(),
        "config": json.dumps(hf_model.config.to_dict(), sort_keys=True),
        "training": hf_model.training,
        "native_generation_config": hf_model.generation_config.to_dict(),
        "shape": (model.n_layers, model.d_model),
        "parameters": tuple(tensor_signature(p) for p in hf_model.parameters()),
        "buffers": tuple(tensor_signature(p) for p in hf_model.buffers()),
        "lens": tuple((layer, tensor_signature(matrix))
                      for layer, matrix in sorted(fitted_lens.jacobians.items())),
        "lens_shape": (fitted_lens.d_model, fitted_lens.n_prompts),
        "lens_file": (str(lens_path), _loaded_lens_sha256),
        "bos": model.bos_policy,
        "tokenizer_specials": str(tokenizer.special_tokens_map),
        "tokenizer_settings": str(tokenizer.init_kwargs),
        "input_device": str(model.input_device),
        "matmul": torch.get_float32_matmul_precision(),
        "tf32": (torch.backends.cuda.matmul.allow_tf32, torch.backends.cudnn.allow_tf32),
    }


def require_loaded():
    message = ("Loaded state is missing or changed. Rerun the model/lens load cell, "
               "then the metadata/cache cell and evaluation cells (and explorer).")
    binding = globals().get("_loaded_binding")
    try:
        valid = binding is not None and all(
            ref() is not None and globals().get(name) is ref()
            for name, ref in binding["objects"].items()
        )
        valid = valid and live_signature() == binding["signature"]
    except Exception as error:
        raise RuntimeError(message) from error
    if not valid:
        raise RuntimeError(message)
    return binding


_loading_config = requested_load_config()
# Explicit action: load model/tokenizer and pretrained lens (no fitting).
hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, dtype=DTYPE,
).to(DEVICE).eval()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
adapter = jlens.from_hf(hf_model, tokenizer, compile=False, force_bos=False)
model = ExplicitPromptModel(adapter, bos_policy=BOS_POLICY)
if model.n_layers < 2:
    raise ValueError("This inner-layer comparison requires at least two model blocks.")

CACHE_DIR = Path(CACHE_DIR).expanduser().resolve()
CACHE_DIR.mkdir(parents=True, exist_ok=True)
if LOCAL_LENS_PATH is not None:
    lens_path = Path(LOCAL_LENS_PATH).expanduser().resolve()
    if not lens_path.is_file():
        raise FileNotFoundError(lens_path)
else:
    if urllib.parse.urlparse(LENS_URL).scheme != "https":
        raise ValueError("Use an HTTPS pretrained lens URL.")
    url_key = hashlib.sha256(LENS_URL.encode()).hexdigest()[:24]
    lens_path = CACHE_DIR / f"lens-{url_key}.pt"
    if REFRESH_LENS or not lens_path.is_file():
        fd, name = tempfile.mkstemp(dir=CACHE_DIR, suffix=".download")
        os.close(fd)
        temporary = Path(name)
        try:
            urllib.request.urlretrieve(LENS_URL, temporary)
            temporary.replace(lens_path)
        finally:
            temporary.unlink(missing_ok=True)

_lens_digest = hashlib.sha256()
with lens_path.open("rb") as handle:
    for chunk in iter(lambda: handle.read(1024 * 1024), b""):
        _lens_digest.update(chunk)
_loaded_lens_sha256 = _lens_digest.hexdigest()
del _lens_digest
# chunk may be absent for an empty/invalid file; avoid retaining its last bytes.
globals().pop("chunk", None)
candidate = jlens.JacobianLens.load(str(lens_path))
required = set(range(model.n_layers - 1))
if candidate.d_model != model.d_model:
    raise ValueError("Lens/model residual widths differ.")
if required - set(candidate.source_layers):
    raise ValueError(f"Missing inner layers: {sorted(required - set(candidate.source_layers))}")
if set(candidate.source_layers) - set(range(model.n_layers)):
    raise ValueError("Lens contains out-of-range layer indices.")
for layer, matrix in candidate.jacobians.items():
    if matrix.shape != (model.d_model, model.d_model) or not torch.isfinite(matrix).all():
        raise ValueError(f"Invalid/nonfinite Jacobian at layer {layer}.")
# Always use the shared model output at the final block, ignoring any stored final J.
fitted_lens = jlens.JacobianLens(
    {layer: candidate.jacobians[layer] for layer in sorted(required)},
    n_prompts=candidate.n_prompts, d_model=candidate.d_model,
)
del candidate, matrix
spellings = DecodedSpellings(
    tokenizer, vocab_size=hf_model.get_output_embeddings().weight.shape[0],
)
if requested_load_config() != _loading_config:
    raise RuntimeError("Configuration changed during loading; rerun load, metadata and evaluation cells.")
_loaded_binding = {
    "objects": {name: weakref.ref(globals()[name]) for name in
                ("model", "adapter", "hf_model", "tokenizer", "fitted_lens", "spellings")},
    "signature": live_signature(),
}
print(adapter)
print("Loaded pretrained lens; rerun metadata/cache and evaluation cells next.")
print("External fitting provenance UNVERIFIED.")

In [ ]:
# Metadata is committed only after the entire cell succeeds.
run_metadata = _metadata_binding = None
require_loaded()


def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def fingerprint(value):
    return hashlib.sha256(
        json.dumps(value, sort_keys=True, ensure_ascii=False).encode()
    ).hexdigest()


def implementation_hashes(*modules):
    # Hash source, including local edits, not only the repository commit.
    return {name: file_sha256(REPO_DIR / "jlens" / f"{name}.py") for name in modules}


def require_run(expected=None):
    loaded = require_loaded()
    binding = globals().get("_metadata_binding")
    if (binding is None or binding["loaded"] is not loaded
            or globals().get("run_metadata") is None
            or fingerprint(run_metadata) != binding["digest"]
            or (expected is not None and binding is not expected)):
        raise RuntimeError("Provenance is stale or missing. Rerun the metadata/cache cell "
                           "and evaluation cells; recreate the explorer before using it.")
    return binding


def cached(namespace, settings, compute, *, refresh=False):
    """Validate live provenance before any cache access and after computation."""
    binding = require_run()
    # Snapshot mutable settings so a computation cannot change its own cache key.
    description = json.loads(json.dumps(
        {"protocol": "pretrained-generated-summary-v1", "run": run_metadata,
         "namespace": namespace, "settings": settings},
    ))
    key = fingerprint(description)
    directory = CACHE_DIR / namespace
    directory.mkdir(exist_ok=True)
    path = directory / f"{key}.pkl"
    if USE_CACHES and not refresh and path.is_file():
        require_run(binding)
        with path.open("rb") as handle:
            payload = pickle.load(handle)  # Never open a downloaded/untrusted pickle.
        require_run(binding)
        if payload["description"] != description:
            raise ValueError(f"Cache provenance mismatch: {path}")
        print(f"Cache hit [{namespace}]: {path.name}")
        return payload["value"]
    value = compute()  # Failed runs never replace a previous valid cache.
    require_run(binding)
    fd, name = tempfile.mkstemp(dir=directory, suffix=".tmp")
    temporary = Path(name)
    try:
        with os.fdopen(fd, "wb") as handle:
            pickle.dump({"description": description, "value": value}, handle)
        require_run(binding)
        temporary.replace(path)
    finally:
        temporary.unlink(missing_ok=True)
    require_run(binding)
    path.with_suffix(".json").write_text(json.dumps(description, indent=2))
    print(f"Cache saved [{namespace}]: {path.name}")
    return value


try:
    repo_commit = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True,
    ).strip()
except (OSError, subprocess.CalledProcessError):
    repo_commit = None
# Include even slow-tokenizer vocab/config; do not silently key all such tokenizers alike.
tokenizer_signature = {
    "vocab": sorted(tokenizer.get_vocab().items()),
    "special_tokens": tokenizer.special_tokens_map,
    "backend": tokenizer.backend_tokenizer.to_str()
        if hasattr(tokenizer, "backend_tokenizer") else str(tokenizer.init_kwargs),
}
run_metadata = {
    "model_id": MODEL_ID, "requested_revision": MODEL_REVISION,
    "resolved_model_commit": getattr(hf_model.config, "_commit_hash", None),
    "model_config_sha256": fingerprint(hf_model.config.to_dict()),
    "tokenizer_class": type(tokenizer).__name__,
    "tokenizer_revision": tokenizer.init_kwargs.get("_commit_hash"),
    "tokenizer_sha256": fingerprint(tokenizer_signature),
    "dtype": str(hf_model.dtype), "device": str(model.input_device),
    "requested_dtype": str(DTYPE), "requested_device": str(DEVICE),
    "effective_parameter_types": sorted({str(p.dtype) for p in hf_model.parameters()}),
    "effective_parameter_devices": sorted({str(p.device) for p in hf_model.parameters()}),
    "accelerator": torch.cuda.get_device_name(model.input_device)
        if torch.device(model.input_device).type == "cuda" else "CPU",
    "bos_policy": model.bos_policy, "preserve_prompt_whitespace": True,
    "lens_sha256": _loaded_lens_sha256,
    "lens_source": str(lens_path) if LOCAL_LENS_PATH is not None else LENS_URL,
    "lens_n_prompts": fitted_lens.n_prompts, "fitting_provenance": "UNVERIFIED",
    "n_layers": model.n_layers, "d_model": model.d_model,
    "scoring": "decoded-whole-token-v1",
    "ranking": "pre-softcap-score-desc-token-id-asc-v1",
    "prediction_space": "native-distribution-logits", "torch": torch.__version__,
    "transformers": transformers.__version__, "repo_commit": repo_commit,
    "matmul_precision": torch.get_float32_matmul_precision(),
    "cuda_matmul_tf32": torch.backends.cuda.matmul.allow_tf32,
    "cudnn_tf32": torch.backends.cudnn.allow_tf32,
    "implementation": implementation_hashes("hf", "readout", "strict_scoring", "lens", "hooks"),
}
require_loaded()
(CACHE_DIR / f"run-{fingerprint(run_metadata)}.json").write_text(
    json.dumps(run_metadata, indent=2),
)
_metadata_binding = {"loaded": require_loaded(), "digest": fingerprint(run_metadata)}
require_run()
display(run_metadata)
print("Refresh flags affect only their named cache; reset them after recomputing.")
print("USE_CACHES=False recomputes and replaces the matching locally generated cache.")
print("For local/mutable model weights without a resolved commit, refresh all result caches after changes.")

## 1. Batched lens measurements

These are the unchanged lexical probes, computed on the explicitly prepared prompts below.
The model's next-token match is retained separately from generated-answer correctness.


In [ ]:
all_words = items = token_items = generations = answers = None
_task_binding = _answer_binding = None


def prepare_tasks():
    if TASK_PROMPT_POLICY not in {"trim_completion_spaces", "preserve"}:
        raise ValueError("Unknown TASK_PROMPT_POLICY")
    prepared = {name: load_eval(str(REPO_DIR), name) for name in DATASETS}
    if TASK_PROMPT_POLICY == "trim_completion_spaces":
        for name in ("multihop", "multilingual", "order-ops"):
            prepared[name] = [
                {**item, "prompt": item["prompt"].rstrip(" \t")}
                for item in prepared[name]
            ]
    return prepared


def current_task_settings():
    return {
        "evals": prepare_tasks(), "batch": BATCH_OPTIONS,
        "prompt_policy": TASK_PROMPT_POLICY,
        "implementation": implementation_hashes("batched_evaluation", "evaluation", "metrics"),
    }


def require_task_results():
    require_run()
    if _task_binding != fingerprint(current_task_settings()) or token_items is None:
        raise RuntimeError("Task configuration changed; rerun lens evaluation, generation and scoring.")


evals = prepare_tasks()
prompt_audit = pd.DataFrame([
    {"dataset": name, "item": original["name"],
     "original": repr(original["prompt"]), "effective": repr(prepared["prompt"])}
    for name in DATASETS
    for original, prepared in zip(load_eval(str(REPO_DIR), name), evals[name], strict=True)
])
display(prompt_audit.groupby("dataset", sort=False).head(2))
task_settings = current_task_settings()
all_words, token_items = cached(
    "tasks", task_settings,
    lambda: evaluate_paired_batched(
        model, fitted_lens, evals, spelling_lookup=spellings,
        preserve_prompt_whitespace=True, **BATCH_OPTIONS,
    ), refresh=REFRESH_TASKS,
)
token_items = token_items.copy()
token_items["next_token_target_match"] = token_items["model_correct"]
items = token_items.copy()
_task_binding = fingerprint(task_settings)
require_task_results()
display(token_items.attrs.get("evaluation", {}))


In [ ]:
require_task_results()
correct_counts = answer_counts(token_items, datasets=DATASETS)
print("Next-token target match only; denominator excludes unsupported token targets.")
display(correct_counts.style.format({"supported_accuracy": "{:.1%}"}, na_rep="N/A"))


## 2. Generate completed answers (cached separately)

Only annotated tasks require generation. Prompts use exactly the same BOS/preprocessing as
lens evaluation. HF generation uses **left padding** and attention masks so each row starts
from its last real prompt token; lens evaluation retains mask-safe right padding. Greedy
batches use KV caching and stop at EOS or `max_new_tokens`. The token budget includes the
maximum continuation. Reduce the batch size/budget if GPU memory is tight. A rerun uses
cached token IDs and decoded text; changing aliases alone never runs the model.


In [ ]:
require_task_results()
generations = answers = None
_answer_binding = None
items = token_items.copy()


def current_generation_settings():
    require_task_results()
    return {
        "evals": prepare_tasks(), "options": GENERATION_OPTIONS,
        "prompt_policy": TASK_PROMPT_POLICY,
        "decoding": greedy_generation_config(
            hf_model, tokenizer, max_new_tokens=GENERATION_OPTIONS["max_new_tokens"],
        ).to_dict(),
        "implementation": {
            "generation": hashlib.sha256((
                inspect.getsource(generate_answers_batched)
                + inspect.getsource(greedy_generation_config)
            ).encode()).hexdigest(),
            **implementation_hashes("strict_scoring"),
        },
    }


generation_settings = current_generation_settings()
generations = cached(
    "generated-answers", generation_settings,
    lambda: generate_answers_batched(hf_model, model, evals, **GENERATION_OPTIONS),
    refresh=REFRESH_GENERATIONS,
)
display(generations.attrs["generation"])


In [ ]:
require_task_results()
answers = None
_answer_binding = None
items = token_items.copy()
if generations is None or fingerprint(generation_settings) != fingerprint(current_generation_settings()):
    raise RuntimeError("Generation is missing or stale; rerun the generation cell.")
answers = score_generations(generations, aliases=ANSWER_ALIASES)
# Verify both provenance fields before merging one answer into both lens rows.
audit = token_items[["dataset", "item", "prompt", "target"]].drop_duplicates().merge(
    answers[["dataset", "item", "prompt", "target"]], on=["dataset", "item"],
    suffixes=("_lens", "_generation"), validate="one_to_one", how="outer", indicator=True,
)
assert audit["_merge"].eq("both").all()
assert audit.prompt_lens.eq(audit.prompt_generation).all()
assert (audit.target_lens.fillna("") == audit.target_generation.fillna("")).all()
items = token_items.merge(
    answers.drop(columns=["prompt", "target"]), on=["dataset", "item"],
    how="left", validate="many_to_one",
)
_answer_binding = fingerprint({
    "generation": generation_settings, "aliases": ANSWER_ALIASES,
    "scoring": SCORING_VERSION, "implementation": implementation_hashes("generated_answers"),
})


def require_answers():
    if answers is None or _answer_binding != fingerprint({
        "generation": current_generation_settings(), "aliases": ANSWER_ALIASES,
        "scoring": SCORING_VERSION, "implementation": implementation_hashes("generated_answers"),
    }):
        raise RuntimeError("Answers are missing/stale; rerun generation if needed, then scoring.")


require_answers()
generated_counts = generated_answer_counts(answers)
print("Generated-answer accuracy: ALL annotated items, including multi-token targets.")
display(generated_counts.style.format({"generated_accuracy": "{:.1%}"}, na_rep="N/A"))
display(answers.groupby(["dataset", "answer_status"], sort=False).size().rename("items").to_frame())
# Inspect actual text instead of inferring model behavior from a single argmax.
inspection = items.drop_duplicates(["dataset", "item"])
display(inspection[inspection.target.notna()].groupby("dataset", sort=False).head(8)[[
    "dataset", "item", "target", "model_top1", "next_token_target_match",
    "generated_text", "generated_answer", "answer_correct", "answer_status",
    "generation_truncated",
]])


## 3. Intermediate retrieval: all items versus generated-answer-correct

Each supported intermediate hits if any inner block has rank ≤ K. Average within each
item, then across eligible items. The right panel selects **completed-answer correctness**,
including items whose target has no single-token spelling. It does not select next-token
matches. Lens rank support/coverage is unchanged. Read eligible counts before comparing
bars; unannotated tasks have no correct-only population.


In [ ]:
require_answers()
intermediate = retrieval_summary(
    all_words, items, kind="intermediate", k=K, layer_scope="inner",
    subsets=("all", "correct"), datasets=DATASETS, correctness_column="answer_correct",
)
coverage_columns = [
    "dataset", "lens", "subset", "score", "n_items_subset", "n_items_used",
    "n_items_excluded", "n_words_total", "n_words_used", "n_words_excluded",
    "word_coverage",
]
display(intermediate[coverage_columns].style.format(na_rep="N/A", precision=3))
fig, axes = plot_retrieval_summary(intermediate)
plt.show()
plt.close(fig)

## 4. Annotated final-answer target retrieval

“Final-answer token” means the **annotated answer target**, not the last input token and
not necessarily the model's predicted token. Compare both lenses' best **inner-layer**
retrieval@K. Separately report target retrieval@1/@K at the **shared final model output**;
there is no duplicate final-output lens bar. Pre-softcap lexical rank@1 can differ from actual
distribution argmax target match after softcap saturation; exact lexical ties use ascending
token ID. These lexical probes use the same IDs as the next-token counts, never a prefix.
They do not measure generated-answer accuracy.
Unannotated or unsupported targets are N/A with explicit counts.

In [ ]:
require_answers()
answer_targets = retrieval_summary(
    all_words, items, kind="target", k=K, layer_scope="inner",
    subsets=("all",), datasets=DATASETS,
)
final_targets = target_final_counts(all_words, items, k=K, datasets=DATASETS)
display(answer_targets[coverage_columns].style.format(na_rep="N/A", precision=3))
# Compact shared-output table: one row/dataset; never duplicate by lens.
shared_final = final_targets.pivot(index="dataset", columns="k", values="hit_rate")
shared_final.columns = [f"shared_output_target@{cutoff}" for cutoff in shared_final.columns]
shared_final = correct_counts.set_index("dataset")[[
    "annotated", "supported", "unsupported", "unannotated",
]].join(shared_final).reindex(DATASETS)
display(shared_final.style.format(na_rep="N/A", precision=3))

fig, ax = plt.subplots(figsize=(10, 3.8))
lens_names = list(answer_targets["lens"].unique())
x = np.arange(len(DATASETS))
width = 0.8 / len(lens_names)
for index, name in enumerate(lens_names):
    scores = answer_targets[answer_targets["lens"].eq(name)].set_index("dataset")["score"].reindex(DATASETS)
    positions = x - 0.4 + width * (index + 0.5)
    ax.bar(positions, scores, width, label=name)  # NaN stays unavailable, not zero.
    for position, score in zip(positions, scores, strict=True):
        if pd.isna(score):
            ax.text(position, 0.02, "N/A", ha="center", fontsize=8, rotation=90)
ax.set_xticks(x, DATASETS, rotation=25, ha="right")
ax.set(ylim=(0, 1), ylabel="Supported-item target hit rate",
       title=f"Annotated answer-target retrieval @{K} — inner layers, all items")
ax.grid(axis="y", alpha=0.2)
ax.legend()
fig.tight_layout()
plt.show()
plt.close(fig)

## 5. Anthropic Figures 52 / 55 / 56 — adaptations, not reproductions

Reference: [Verbalizable Representations Form a Global Workspace](https://transformer-circuits.pub/2026/workspace/index.html).
Model, pretrained lens, spelling protocol, and evaluation samples differ. No tuned lens,
workspace shading, matching-pretraining-distribution claim, or uncertainty estimate is implied.
See `docs/reference_plots.md` and `docs/lens_metrics.md` for formulas; this notebook overrides
implicit whitespace stripping/prefix scoring with the explicit protocol above.

**52:** item-weighted intermediate rank sweep, best over **all blocks, including the shared
final output** (unlike our primary inner-only metric). Unsupported annotation rows must be
passed through so exclusion counts are retained. This plot reuses the task cache only.

In [ ]:
require_answers()
sweep = intermediate_rank_sweep(all_words)
display(sweep.counts)
fig52, axes52 = plot_intermediate_sweep(sweep)
plt.show()
plt.close(fig52)

### Independent held-out diagnostics (optional)

Set `HELD_OUT_TEXTS` and `HELD_OUT_SOURCE` in configuration, then rerun this cell. Supply
independent passages with recorded selection/split; do not use task prompts or knowingly
reuse fitting texts. External lens corpus overlap cannot be ruled out from its legacy file.
The default **skips** both figures and clears old outputs/state. This is extra **batched**
inference, never fitting or serial fallback. Overlength passages fail instead of truncating.

**55:** token-weighted KL(model ‖ lens), entropy (nats), and top-1 agreement with model
(percent) at nonpadding, nonspecial input positions. Agreement is not task correctness.
**56:** same-layer J/logit symmetric KL = half-sum, top-1 agreement (fraction), plus
vocabulary-vector cosine of the **linear heads only** (normalization/bias/softcap excluded
from geometry, not probabilistic readouts). Unsupported geometry is N/A. Depth uses actual
block indices; both lenses share the final output. Held-out metrics and geometry have
independent persistent caches. No model pass is needed to redraw cached tables.

In [ ]:
# Keep computation and rendering together: disabling/failing cannot show an old figure.
clear_output(wait=False)
held_out_metrics = held_out_geometry = held_out_run = None
fig55 = fig56 = None
if HELD_OUT_TEXTS is None:
    print("Figures 55/56 skipped: supply independent HELD_OUT_TEXTS and HELD_OUT_SOURCE.")
else:
    if not isinstance(HELD_OUT_TEXTS, (list, tuple)) or not HELD_OUT_TEXTS:
        raise ValueError("Supply a nonempty list/tuple of independent strings.")
    if not all(isinstance(text, str) and text.strip() for text in HELD_OUT_TEXTS):
        raise ValueError("Every passage must be a nonempty string; original whitespace is preserved.")
    if not isinstance(HELD_OUT_SOURCE, str) or not HELD_OUT_SOURCE.strip():
        raise ValueError("Describe corpus/version, split, selection and independence first.")
    task_prompts = {entry["prompt"] for group in evals.values() for entry in group}
    if any(text in task_prompts for text in HELD_OUT_TEXTS):
        raise ValueError("Held-out texts must not be task prompts.")
    from jlens.metrics import lens_vector_geometry

    held_out_run = {
        "source": HELD_OUT_SOURCE, "texts": list(HELD_OUT_TEXTS),
        "options": HELD_OUT_OPTIONS, "layers": "all", "pairwise": True,
        "weighting": "nonpadding nonspecial input tokens",
        "implementation": implementation_hashes("metrics", "batched_evaluation"),
    }
    candidate_metrics = cached(
        "held-out", held_out_run,
        lambda: evaluate_distributions_batched(
            model, fitted_lens, HELD_OUT_TEXTS, **HELD_OUT_OPTIONS,
        ), refresh=REFRESH_HELD_OUT,
    )
    candidate_geometry = cached(
        "geometry", {
            "vocab_chunk_size": GEOMETRY_VOCAB_CHUNK_SIZE,
            "layers": "all", "implementation": implementation_hashes("metrics"),
        },
        lambda: lens_vector_geometry(
            model, fitted_lens, layers=None,
            vocab_chunk_size=GEOMETRY_VOCAB_CHUNK_SIZE,
        ), refresh=REFRESH_GEOMETRY,
    )
    held_out_metrics, held_out_geometry = candidate_metrics, candidate_geometry
    display({key: value for key, value in held_out_run.items() if key != "texts"})
    display(held_out_metrics.layers[["n_texts", "n_texts_used", "n_tokens", "weighting"]].drop_duplicates())
    display(held_out_metrics.layers.attrs.get("evaluation", {}))
    display(held_out_geometry.groupby("status", dropna=False).size().rename("layers").to_frame())
    unavailable_geometry = held_out_geometry[held_out_geometry["status"].ne("ok")]
    if not unavailable_geometry.empty:
        display(unavailable_geometry[["layer", "status", "reason", "n_valid_vectors", "n_zero_vectors"]])
    fig55, axes55 = plot_distribution_summary(held_out_metrics.layers, n_layers=model.n_layers)
    plt.show()
    plt.close(fig55)
    fig56, axes56 = plot_lens_comparison(
        held_out_metrics.pairs, held_out_geometry, n_layers=model.n_layers,
    )
    plt.show()
    plt.close(fig56)

## Conclusions and limits

The offline examples verify that complete numeric answers such as `14` and `fourteen`
match, while `1`, `140` and a later mention after a wrong answer do not. They are software
checks, not Gemma results. Run the GPU cells before concluding which lens performs better.

Report generated accuracy over all annotated items separately from supported next-token
matches. Compare J/logit retrieval with coverage and correct-only counts. Better retrieval
cannot improve the unchanged model's answer. Generated-answer scoring is deliberately
conservative: no reasoning-trace extraction, later correction, fuzzy matching, or automatic
synonym inference. Inspect false negatives and truncations; adjust explicit aliases and
rescore cached completions if warranted, reporting those changes.

The raw-completion format differs from an instruction-tuned chat benchmark. Token budget,
BOS/whitespace policy, model revision, precision, and unknown fitting provenance limit
comparisons. Lens ranks remain single-token probes and do not prove causal use of a concept.
No held-out Figures 55/56 are available without independent texts. Save executed model
outputs and manifests after running; the original notebooks remain unchanged.


## Explore one prompt — explicit run action, no startup inference

Edit the prompt below (trailing whitespace matters), then click **Run J-lens slice**.
Exactly one requested prompt is computed per action; no loop over a preset list. Repeated
identical prompts/settings use the slice cache. The button is disabled while processing;
errors clear the previous display. With no working widget frontend, edit the same Python
cell and set `RUN_SLICE_NOW=True`, then execute it; reset the flag afterward.

The actual `jlens.vis` iframe is rendered, with the same model/BOS policy. `mask_display`
only selects word-like displayed candidates; ranks remain full-vocabulary. Optional
`assets/qwen_gloss.json.gz` labels follow `walkthrough_orig`: integer token-ID keys passed
as `alt_token` to `build_page`. They never affect tokens or scoring; their tokenizer provenance
is unverified. Glosses are disabled for replacement models even if requested. No content
or analysis follows this explorer. The first embedded page render fetches integrity-checked
D3 from jsDelivr; cached pages are self-contained thereafter.

In [ ]:
SLICE_PROMPT = "2 * (3 + 4) ="  # Edit freely; never stripped.
RUN_SLICE_NOW = False  # Editable-cell fallback; True runs only this requested prompt.
SLICE_OPTIONS = dict(
    top_n=10, max_tracked=256, layer_stride=1, last_n_tokens=32,
    max_seq_len=512, mask_display=True,
)

close_slice_explorer()
clear_output(wait=False)
_explorer_binding = require_run()


def run_slice(prompt, _expected=_explorer_binding):
    """One explicit prompt; reject callbacks bound to an older loaded run."""
    require_run(_expected)
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError("Enter a nonempty prompt.")
    # Validate without stripping or truncating, even when a page cache exists.
    model.encode_ids(prompt, max_length=SLICE_OPTIONS["max_seq_len"])
    gloss = None
    gloss_path = REPO_DIR / "assets" / "qwen_gloss.json.gz"
    gloss_hash = None
    if ENABLE_QWEN_GLOSS and MODEL_ID == "Qwen/Qwen3.5-9B-Base":
        if not gloss_path.is_file():
            raise FileNotFoundError(f"Optional display glossary missing: {gloss_path}")
        gloss_hash = file_sha256(gloss_path)
        with gzip.open(gloss_path, "rt", encoding="utf-8") as handle:
            gloss = {int(key): value for key, value in json.load(handle).items()}
        print("Display-only Qwen glossary enabled; token-ID provenance unverified.")
    elif ENABLE_QWEN_GLOSS:
        print("Qwen glossary disabled: this is a replacement model.")

    settings = {
        "prompt": prompt, "options": SLICE_OPTIONS, "gloss_sha256": gloss_hash,
        "implementation": implementation_hashes("vis"),
        "vis_assets": {
            str(path.relative_to(REPO_DIR)): file_sha256(path)
            for directory in (REPO_DIR / "assets", REPO_DIR / "jlens" / "data")
            for path in sorted(directory.rglob("*"))
            if path.is_file() and path.suffix in {".html", ".js", ".css"}
        },
    }

    def compute_page():
        with torch.inference_mode():
            data = compute_slice(model, fitted_lens, prompt, **SLICE_OPTIONS)
        page, _, _ = build_page(
            data, prompt, title=f"{MODEL_ID} — pretrained J-lens",
            description="Full-vocabulary ranks; optional glosses are display only.",
            alt_token=gloss,
        )
        return page

    page = cached("slices", settings, compute_page, refresh=REFRESH_SLICES)
    display(notebook_iframe(page))


try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

if widgets is not None:
    slice_prompt_editor = widgets.Textarea(
        value=SLICE_PROMPT, description="Prompt:",
        layout=widgets.Layout(width="100%", height="100px"),
    )
    slice_run_button = widgets.Button(description="Run J-lens slice", button_style="primary")
    slice_output = widgets.Output()

    def on_slice_click(_, _expected=_explorer_binding):
        require_run(_expected)
        slice_run_button.disabled = True
        try:
            with slice_output:
                clear_output(wait=False)
                try:
                    run_slice(slice_prompt_editor.value)
                except Exception as error:
                    print(f"Slice failed: {type(error).__name__}: {error}")
        finally:
            slice_run_button.disabled = False

    slice_run_button.on_click(on_slice_click)
    slice_ui = widgets.VBox([slice_prompt_editor, slice_run_button, slice_output])
    display(slice_ui)
    if RUN_SLICE_NOW:
        on_slice_click(None)
else:
    print("Widgets unavailable. Edit SLICE_PROMPT, set RUN_SLICE_NOW=True, and rerun this cell.")
    if RUN_SLICE_NOW:
        run_slice(SLICE_PROMPT)